**Projeto 2 - Tradução Automática Usando Atenção**

*MS671 - Introdução ao Aprendizado de Máquinas Profundo*


---


**Alunos:**

*   *Amanda Rehder Cedro: 253507*
*   *Elisa Amorim Da Costa: 277145*
*   *Iuri Santos Oliveira: 194610*
*   *Pedro Galhardo Barbosa: 277218*

In [1]:
#Parte 1 — Clonar o repositório
import os

REPO_NAME = "Deep_Learning_P2"
REPO_URL = "https://github.com/your_username/Deep_Learning_P2.git"  # Define REPO_URL here
BASE_PATH = f"/content/{REPO_NAME}/arquivos"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

BASE_PATH = f"/content/{REPO_NAME}/arquivos"

#Parte 2 — Importações
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import *
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Softmax
from tensorflow.keras.optimizers import Adam
import pickle

#Parte 3 — Carregar arquivos
with open(f"{BASE_PATH}/dataset.pkl", "rb") as f:
    dataset = pickle.load(f)

with open(f"{BASE_PATH}/human_vocab.pkl", "rb") as f:
    human_vocab = pickle.load(f)

with open(f"{BASE_PATH}/machine_vocab.pkl", "rb") as f:
    machine_vocab = pickle.load(f)

with open(f"{BASE_PATH}/inv_machine_vocab.pkl", "rb") as f:
    inv_machine_vocab = pickle.load(f)

#Parte 4 — Conversão do dataset
Tx = 30
Ty = 10
m = len(dataset)
X = np.array(
    [
        string_to_int(human, Tx, human_vocab)
        for human, machine in dataset
    ]
)
Y = np.array(
    [
        string_to_int(machine, Ty, machine_vocab)
        for human, machine in dataset
    ]
)
Xoh = np.array(
    list(map(
        lambda x: to_categorical(x, num_classes=len(human_vocab)),
        X
    ))
)
Yoh = np.array(
    list(map(
        lambda x: to_categorical(x, num_classes=len(machine_vocab)),
        Y
    ))
)
Yoh = np.swapaxes(Yoh,0,1)

#Parte 5 — Camadas globais da atenção
#parâmetros padrão
n_a = 32
n_s = 64
repeator = RepeatVector(Tx)
concatenator = Concatenate(axis=-1)
densor1 = Dense(10, activation="tanh")
densor2 = Dense(1, activation="relu")
activator = Softmax(
    axis=1,
    name='attention_weights'
)
dotor = Dot(axes=1)

#Função one step attention
def one_step_attention(a, s_prev):
    s_prev = repeator(s_prev)
    concat = concatenator([a, s_prev])
    e = densor1(concat)
    energies = densor2(e)
    alphas = activator(energies)
    context = dotor([alphas, a])
    return context

#Parte 6 — Construção do modelo
output_layer = Dense(
    len(machine_vocab),
    activation='softmax'
)
def modelf(Tx, Ty, n_a, n_s,
           human_vocab_size,
           machine_vocab_size):
    X = Input(shape=(Tx, human_vocab_size))
    s0 = Input(shape=(n_s,), name='s0')
    c0 = Input(shape=(n_s,), name='c0')
    s = s0
    c = c0
    outputs = []
    a = Bidirectional(
        LSTM(
            n_a,
            return_sequences=True
        )
    )(X)

    # Define post_activation_LSTM_cell here to be part of the model's graph
    _post_activation_LSTM_cell = LSTM(
        n_s,
        return_state=True
    )

    for t in range(Ty):
        context = one_step_attention(a, s)
        s, _, c = _post_activation_LSTM_cell(
            context,
            initial_state=[s,c]
        )
        out = output_layer(s)
        outputs.append(out)
    model = Model(
        inputs=[X,s0,c0],
        outputs=outputs
    )
    return model

# Parte 7 — Criar modelo
model = modelf(
    Tx,
    Ty,
    n_a,
    n_s,
    len(human_vocab),
    len(machine_vocab)
)
model.summary()

#Parte 8 — Compilar
opt1 = Adam(
    learning_rate=0.005,
    beta_1=0.9,
    beta_2=0.999
)

model.compile(
    optimizer=opt1,
    loss='categorical_crossentropy',
    metrics=['accuracy']*Ty
)

#Parte 9 — Estados iniciais
s0 = np.zeros((m,n_s))
c0 = np.zeros((m,n_s))

#Treinamento por 1 época
history1 = model.fit(
    [Xoh,s0,c0],
    list(Yoh),
    epochs=1,
    batch_size=100
)


#Treinamento por 50 épocas
model50 = modelf(
    Tx,
    Ty,
    n_a,
    n_s,
    len(human_vocab),
    len(machine_vocab)
)
opt2 = Adam(
    learning_rate=0.005,
    beta_1=0.9,
    beta_2=0.999
)

model50.compile(
    optimizer=opt2,
    loss='categorical_crossentropy',
    metrics=['accuracy']*Ty
)
history50 = model50.fit(
    [Xoh,s0,c0],
    list(Yoh),
    epochs=50,
    batch_size=100
)

#Parte 10 — Carregar modelo pré-treinado
# Load with custom_objects to handle functional API layers correctly
pretrained_model = tf.keras.models.load_model(
    f"{BASE_PATH}/model.h5",
    custom_objects={
        'softmax': Softmax,
        'Input': Input,
        'Bidirectional': Bidirectional,
        'LSTM': LSTM,
        'RepeatVector': RepeatVector,
        'Concatenate': Concatenate,
        'Dense': Dense,
        'Dot': Dot
    }
)


#Parte 11 — Função de inferência
def int_to_string(ints, inv_vocab):
    return ''.join(
        inv_vocab[i]
        for i in ints
    )
def predict_date(modelx, text):
    source = string_to_int(
        text,
        Tx,
        human_vocab
    )
    source = np.array(source)
    source = np.array(
        list(
            map(
                lambda x:
                to_categorical(
                    x,
                    num_classes=len(human_vocab)
                ),
                source
            )
        )
    )
    source = source.reshape(
        1,
        Tx,
        len(human_vocab)
    )
    s0 = np.zeros((1,n_s))
    c0 = np.zeros((1,n_s))
    prediction = modelx.predict(
        [source,s0,c0],
        verbose=0
    )
    prediction = np.argmax(
        prediction,
        axis=-1
    )
    output = ''.join(
        inv_machine_vocab[int(i)]
        for i in prediction[:,0]
    )
    return output


#Parte 12 — Exemplos
examples = [
    "17th of October 2023",
    "Tuesday October 17 2023",
    "3 May 1979",
    "Saturday May 9 2018",
    "1st January 2000"
]

#Modelo 1 época
print("MODELO 1 ÉPOCA\n")
for ex in examples:
    print(ex)
    print(predict_date(model, ex))
    print()


#Modelo 50 épocas
print("MODELO 50 ÉPOCAS\n")
for ex in examples:
    print(ex)
    print(predict_date(model50, ex))
    print()


#Modelo pré-treinado
print("MODELO PRÉ-TREINADO\n")
for ex in examples:
    print(ex)
    print(predict_date(pretrained_model, ex))
    print()


#Parte 13 — Testar diferentes n_a e n_s
#configuração 1
n_a = 16
n_s = 32
#configuração 2
n_a = 32
n_s = 64
#configuração 3
n_a = 64
n_s = 128


#Parte 14 — Mapas de atenção
def plot_attention_map(
    modelx,
    human_vocab,
    inv_machine_vocab,
    input_text,
    n_s=64,
    Tx=30
):
    # Prepare the input text
    source = string_to_int(input_text, Tx, human_vocab)
    source = np.array(source)
    source = np.array(
        list(
            map(
                lambda x:
                to_categorical(
                    x,
                    num_classes=len(human_vocab) # Use len(human_vocab) instead of hardcoded 37
                ),
                source
            )
        )
    )
    source = source.reshape(1, Tx, len(human_vocab))

    # Get initial states
    s0 = np.zeros((1, n_s))
    c0 = np.zeros((1, n_s))

    # Placeholder messages for attention map functionality
    print("Plotting attention map functionality requires further model modification to output attention weights.")
    print("Currently, the model outputs only the predicted sequence.")

plot_attention_map(
    pretrained_model,
    human_vocab,
    inv_machine_vocab,
    "Tuesday October 17 2023"
)

Cloning into 'Deep_Learning_P2'...
fatal: could not read Username for 'https://github.com': No such device or address


FileNotFoundError: [Errno 2] No such file or directory: '/content/Deep_Learning_P2/arquivos/dataset.pkl'